In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)


# Improvements at 25 labels per class

All results are **development evidence**. No untouched test set, new annotations, building footprints or confirmed scale/offset were supplied. Read [the result report](../../docs/RESULTS.md) for outcomes and remaining external requirements.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from scripts.improve_25 import OUT, run
from scripts.verify_improvement_25 import verify

# True retrains all 60 runs; preserves older experiment folders.
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    run()
assert (OUT / "summary.csv").is_file(), "Run python -m scripts.improve_25 first."
print(verify())

## 1. Inspect all candidates and spatial sensitivity

Average repetitions within folds first. Fold SD is not a confidence interval. Compare methods within each regime: buffering changes available adaptation samples, and larger blocks change evaluation membership. The `inner_selected` method was chosen using adaptation labels only; never choose a final method from the highest displayed outer score.


In [ ]:
summary = pd.read_csv(OUT / "summary.csv")
display(summary.round(4))
display(pd.read_csv(OUT / "paired_uncertainty.csv").round(4))

## 2. Class performance, calibration and abstention

All methods have the same 25/class budget within a run. Abstention thresholds use inner OOF predictions, not outer targets. Accepted-only metrics must always be read alongside actual coverage. Balanced calibration samples may differ from deployment class prevalence.


In [ ]:
classes = pd.read_csv(OUT / "class_metrics.csv")
display(classes.loc[classes.regime.eq("buffer2")].groupby(
    ["method", "age_class"])[["precision", "recall", "f1"]].mean().round(4))
display(pd.read_csv(OUT / "operational.csv").groupby("regime")
        [["coverage", "macro_f1", "accuracy", "severe_errors"]].mean().round(4))

## 3. Acquisition quality and label-review queue

Audit raw measurements without inventing scale/offset or QA bit semantics. The tokenized queue identifies spatially diverse difficult cases for human annotation; predictions do not replace ground truth. Keep new adaptation and final-test labels separate.


In [ ]:
display(pd.read_csv(OUT / "acquisition_quality.csv"))
print(json.dumps(json.loads((OUT / "metadata_confirmation.json").read_text()), indent=2))
display(pd.read_csv(OUT / "label_review_queue.csv").head())

## 4. Explanation and independent-evaluation status

Optional grouped SHAP checks explain a fixed selected demonstration model using training backgrounds at multiple sizes/seeds. They establish neither causal importance nor predictive improvement.

The independent evaluator and monitor are in `scripts/independent_evaluation.py`. Complete its data contract with data-owner evidence before using a new feature dataset. Existing development locations are rejected for final evaluation.


In [ ]:
if (OUT / "shap_checks.json").is_file():
    print((OUT / "shap_checks.json").read_text())
if (OUT / "frozen_candidate.json").is_file():
    frozen = json.loads((OUT / "frozen_candidate.json").read_text())
    print({k: frozen[k] for k in ["model", "target_labels",
          "selection", "independent_test_performed"]})